# 20 — Sensibilité au ratio de coût HF:BF

Analyse post-hoc : recalcule coût *données* et coût *total* sous plusieurs ratios HF:BF (modèle résolution² recalibré, `src/cost.py`), pour les 3 datasets × tous les modèles. Vérifie la robustesse du classement par efficacité (précision HF par CA).

## 1) Chargement + modèle de coût

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

for _p in ['../src', 'src', './src', '../../src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import importlib
import analysis_utils as A
importlib.reload(A)

ROOT = A.find_results_root()
OUT = os.path.join(ROOT, "analysis")
os.makedirs(OUT, exist_ok=True)
df = pd.DataFrame(A.load_rows(ROOT))
print("RESULTS_ROOT =", ROOT)
print(f"{len(df)} entrees | datasets: {sorted(df.dataset.unique())} | modeles: {df.model.nunique()}")
import cost as COST
importlib.reload(COST)
RATIOS = [2, 5, 10, 20, 50]
print("Ratios testés :", RATIOS)

RESULTS_ROOT = C:\VSCode\PF22_MultiFidelity_DL\results_gpu_run
33 entrees | datasets: ['Animals-10', 'Imagewoof', 'Intel'] | modeles: 11
Ratios testés : [2, 5, 10, 20, 50]


## 2) Coût données et efficacité par ratio

Le coût *données* (acquisition) ne dépend que du nombre d'images HF/BF distinctes : il est donc **identique** pour toutes les méthodes utilisant le pool HF+BF, et ne diffère qu'entre baselines (HF seul / BF seul / mixte). *(La décomposition par domaine du coût total — images vues HF/BF — n'a pas été journalisée pour toutes les stratégies, on analyse donc le coût données.)*

In [2]:
recs = []
for r in A.load_rows(ROOT):
    if r["n_hf_pool"] is None or r["n_bf_pool"] is None:
        continue
    for R in RATIOS:
        c_hf = COST.unit_cost_ratio(None, R)
        c_bf = COST.unit_cost_ratio(64, R)
        data_c = r["n_hf_pool"] * c_hf + r["n_bf_pool"] * c_bf
        recs.append({
            "dataset": r["dataset"], "model": r["model"], "color": r["color"],
            "ratio": R, "HF": r["HF"], "data_cost": data_c,
            "eff_data": r["HF"] / (data_c / 1000.0) if data_c else np.nan,
        })
cr = pd.DataFrame(recs)
print(cr.shape, "lignes")

(165, 7) lignes


## 3) Efficacité (précision HF par 1000 CA données) en fonction du ratio

In [3]:
fig, axes = plt.subplots(1, len(A.DATASETS), figsize=(6 * len(A.DATASETS), 5))
if len(A.DATASETS) == 1:
    axes = [axes]
for ax, ds in zip(axes, A.DATASETS):
    sub = cr[cr.dataset == ds]
    for model, g in sub.groupby("model"):
        ax.plot(g.ratio, g.eff_data, marker="o", color=g.color.iloc[0], label=model, linewidth=1.3)
    ax.set_xscale("log"); ax.set_xticks(RATIOS); ax.set_xticklabels(RATIOS)
    ax.set_title(ds); ax.set_xlabel("Ratio HF:BF"); ax.set_ylabel("HF par 1000 CA (données)")
    ax.grid(alpha=0.3)
axes[-1].legend(fontsize=6, ncol=1, loc="best")
fig.suptitle("Efficacité (coût données) vs ratio HF:BF", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(os.path.join(OUT, "ratio_efficacite_donnees.png"), dpi=160, bbox_inches="tight")
plt.show()

C:\Users\ivann\AppData\Local\Temp\ipykernel_27264\1817124676.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4) Meilleur modèle (efficacité données) par ratio

In [4]:
summary = {}
for ds in A.DATASETS:
    best = {}
    for R in RATIOS:
        s = cr[(cr.dataset == ds) & (cr.ratio == R)]
        if s.empty:
            continue
        best[R] = s.loc[s.eff_data.idxmax(), "model"]
    robust = len(set(best.values())) == 1
    summary[ds] = {"best_per_ratio": best, "robust": robust}
    print(f"{ds:11s}: " + " | ".join(f"R={R}:{best[R]}" for R in best) +
          f"   -> {'ROBUSTE' if robust else 'sensible au ratio'}")
with open(os.path.join(OUT, "cost_ratio_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print("Sauvé:", os.path.join(OUT, "cost_ratio_summary.json"))

Animals-10 : R=2:BL1 (HF) | R=5:BL1 (HF) | R=10:BL2 (BF) | R=20:BL2 (BF) | R=50:BL2 (BF)   -> sensible au ratio
Imagewoof  : R=2:BL1 (HF) | R=5:BL1 (HF) | R=10:BL2 (BF) | R=20:BL2 (BF) | R=50:BL2 (BF)   -> sensible au ratio
Intel      : R=2:BL1 (HF) | R=5:BL1 (HF) | R=10:BL2 (BF) | R=20:BL2 (BF) | R=50:BL2 (BF)   -> sensible au ratio
Sauvé: C:\VSCode\PF22_MultiFidelity_DL\results_gpu_run\analysis\cost_ratio_summary.json


## Note

Le coût *données* étant identique pour toutes les méthodes HF+BF, leurs courbes d'efficacité sont parallèles (séparées seulement par la précision). Le ratio HF:BF ne fait basculer le classement qu'entre **baselines** (HF seul devient moins efficace que BF seul quand la donnée HF devient chère). Sur la **précision HF absolue**, le choix de stratégie reste indépendant du ratio.